# ST Score Restore — Stage 11 V2d Colab GPU Detector Benchmark

**HOTFIX v3:** Oemer'in eski ONNX modelleri için ONNX Runtime `1.20.1` sabitlendi ve mevcut 20-sayfa GPU Restore cache'inden devam yolu eklendi.

Restore modeli GPU'da kalır. Oemer detectorı uyumluluk ve deterministik preflight için CPU provider kullanır. Bu notebook training/fine-tuning yapmaz; yalnız development corpus üzerinde inference-only benchmark çalıştırır. Final canonical CPU doğrulaması ayrıca gereklidir.

In [ ]:
# 1) GPU + pinned Oemer compatibility runtime
!nvidia-smi || true
!apt-get -qq update
!apt-get -qq install -y poppler-utils
# Oemer's legacy tf2onnx graphs are rejected by new ORT shape inference.
# Use the Python-3.13-compatible 1.20.1 CPU wheel for detector inference only.
!pip -q uninstall -y onnxruntime onnxruntime-gpu >/dev/null 2>&1 || true
!pip -q install "onnxruntime==1.20.1" scipy scikit-learn matplotlib pillow typing-extensions
!pip -q install --no-deps "git+https://github.com/BreezeWhite/oemer@dbe2a933d630d0f74805d717960eb259473f5978"

import sys, subprocess, torch
print('python', sys.version)
print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU runtime gerekli: Colab > Runtime > Change runtime type > GPU')
# Verify ORT in a fresh subprocess so this cell also repairs an already-failed runtime
# where a newer onnxruntime module may still exist in the notebook kernel memory.
subprocess.run([sys.executable, '-c', "import onnxruntime as o; print('detector ORT', o.__version__, o.get_available_providers()); assert o.__version__ == '1.20.1'; assert 'CPUExecutionProvider' in o.get_available_providers()"], check=True)

In [ ]:
# 2) Mount Drive + refresh repository source.
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, sys
from pathlib import Path
REPO = Path('/content/st-score-restore-engine')
if REPO.exists():
    shutil.rmtree(REPO)
!git clone -q -b stage11-v2c-semantic-detector-corpus-expansion https://github.com/khfy7wpr5p-maker/st-score-restore-engine.git /content/st-score-restore-engine
sys.path.insert(0, str(REPO/'src'))
print('repo ready', REPO)
print('source path injected', REPO/'src')

In [ ]:
# 3) Resume safely if the previous run already completed 20/20 GPU Restore pages.
# Otherwise execute the full benchmark. The resume path re-verifies all exact bytes
# and performs an ONNX session preflight before any detector loop.
import os, subprocess, sys
from pathlib import Path
cache_source = Path('/content/v2d_work/source')
cache_restored = Path('/content/v2d_work/restored_gpu_exploratory')
cache_exact = Path('/content/v2d_exact_bytes')
cache_ready = (
    cache_exact.exists() and cache_source.exists() and cache_restored.exists()
    and len(list(cache_source.glob('*.png'))) == 20
    and len(list(cache_restored.glob('*.png'))) == 20
)
print('cached 20-page Restore pass available:', cache_ready)
if cache_ready:
    env = os.environ.copy()
    env['PYTHONPATH'] = str(REPO/'src') + os.pathsep + env.get('PYTHONPATH', '')
    code = "from st_score_restore.stage11_v2d_colab_resume import run_cached_detector_benchmark; p=run_cached_detector_benchmark(); print('DONE:', p)"
    subprocess.run([sys.executable, '-c', code], check=True, env=env)
else:
    # Fresh runtime: ORT has not been imported before the pin above, so full runner is safe.
    from st_score_restore.stage11_v2d_colab_runner import run_colab_benchmark
    out = run_colab_benchmark()
    print('DONE:', out)

## Bittiğinde

Önce `Oemer preflight PASS` satırları, sonra detector ilerlemesi görünmeli. Sonunda `SAVED:` ve `DONE:` satırları çıkmalıdır. Sonuç Google Drive'da `ST_SCORE_RESTORE_STAGE11_EVAL/V2D_RESULTS/v2d_colab_gpu_detector_benchmark_result.json` olarak kaydedilir.

Bu hotfix Oemer modelini değiştirmez; yalnız uyumlu ONNX Runtime sürümünü sabitler.